In [ ]:
import json
import unicodedata
from pathlib import Path

import pandas as pd


In [2]:
ds = pd.read_csv('data/raw/doda_train.csv')
ds.head()


,Darija_latin,Darija_arabic
0,"homa mkhbbyin chi haja, ana mti99en!","هوما مخبّيين شي حاجة, أنا متيقّن!"
1,bayna homa tay7awlo ib9aw mbrrdin.,باينا هوما تايحاولو إبقاو مبرّدين.
2,loTilat mabaynach fihom mori7in bzzaf.,لوطيلات مابايناش فيهوم موريحين بزّاف.
3,ghaliban ayejriw elih men lfou9,غاليبان غايجرّيو عليه من اللفوق
4,Tab3an rah mkta2eb!,طابعان راه مكتاءب!


In [4]:
ds.shape

(10000, 2)

In [5]:
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Darija_latin   10000 non-null  object
 1   Darija_arabic  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [ ]:
clean_ds = ds.dropna(subset=["Darija_latin", "Darija_arabic"]).copy()


In [ ]:
# Normalize Unicode and collapse whitespace before checking for duplicates.
def normalize_text(text):
    return " ".join(unicodedata.normalize("NFC", text).split())

for column in ["Darija_latin", "Darija_arabic"]:
    clean_ds[column] = clean_ds[column].apply(normalize_text)


In [14]:

clean_ds = clean_ds.replace({"Darija_latin": {"": pd.NA},"Darija_arabic": {"": pd.NA}})

In [15]:
clean_ds = clean_ds.dropna(subset=["Darija_latin", "Darija_arabic"])

In [16]:
clean_ds = clean_ds.drop_duplicates(subset=["Darija_latin", "Darija_arabic"])

In [18]:
conflicts = clean_ds.groupby("Darija_latin")["Darija_arabic"].nunique()
conflicts = conflicts[conflicts > 1]

In [ ]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
clean_ds.to_csv(output_dir / "doda_train_clean.csv", index=False)


In [ ]:
# Shuffle unique Latin inputs, then assign each input and all its references
# to one split. Ratios are approximate because groups may have multiple rows.
seed = 42
groups = clean_ds["Darija_latin"].drop_duplicates().sample(frac=1, random_state=seed).tolist()
n_train = int(len(groups) * 0.8)
n_val = int(len(groups) * 0.1)
split_groups = {
    "train": set(groups[:n_train]),
    "val": set(groups[n_train:n_train + n_val]),
    "test": set(groups[n_train + n_val:]),
}

split_data = clean_ds.reset_index(drop=True).copy()
split_data.insert(0, "example_id", split_data.index)
manifest = {"seed": seed, "group_by": "Darija_latin", "splits": {}}
for name, inputs in split_groups.items():
    part = split_data[split_data["Darija_latin"].isin(inputs)]
    part.to_csv(output_dir / f"{name}.csv", index=False)
    manifest["splits"][name] = part["example_id"].tolist()

with (output_dir / "split_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print({name: len(ids) for name, ids in manifest["splits"].items()})
